# Parallel quicksort benchmarking

### Setup

In [1]:
import platform
import re
import subprocess
from datetime import datetime
from pathlib import Path
from typing import Optional, Sequence

REPO_ROOT = Path.cwd()
BINARY = REPO_ROOT / 'src' / 'parallelQuicksort'
DATA_ROOT = REPO_ROOT / 'data'

# CSV column -> label the binary prints for that sort; also fixes the column order.
SORTS = {
	'Seq': 'Sequential',
	'Par': 'Parallel',
	'Libc': 'Built-in',
}

TIMING = re.compile(r'^(\S+) quicksort took: ([\d.]+) sec\.$')


def machine_name() -> str:
	'''The DNS hostname follows whichever network we are on, so on macOS take the
	stable local name instead to keep one directory per machine, not per network.'''
	if platform.system() != 'Darwin':
		return platform.node()

	local_name = subprocess.run(
		['scutil', '--get', 'LocalHostName'],
		capture_output=True, text=True, check=True,
	).stdout.strip()
	return f'{local_name}.local'


def time_quicksorts(size: int) -> dict:
	output = subprocess.run(
		[BINARY, str(size)],
		capture_output=True, text=True, check=True,
	).stdout
	timings = dict(TIMING.match(line).groups() for line in output.splitlines())
	return {
		'Size': size,
		**{column: float(timings[label]) for column, label in SORTS.items()},
	}


def write_wide_csv(records: list, output_path: Path) -> None:
	output_path.parent.mkdir(parents=True, exist_ok=True)
	with output_path.open('w') as csv_file:
		csv_file.write(', '.join(['Size', *SORTS]) + '\n')
		for record in records:
			row = [str(record['Size']), *(f'{record[column]:.6f}' for column in SORTS)]
			csv_file.write(', '.join(row) + '\n')


def run_benchmark(
	sizes: Sequence[int] = (100, 1_000, 10_000, 100_000, 1_000_000),
	reps: int = 5,
	output_path: Optional[Path] = None,
) -> list:
	if output_path is None:
		now = datetime.now()
		output_path = (
			DATA_ROOT
			/ f'{machine_name()}_{now:%Y-%m-%d}'
			/ f'measurements_{now:%H:%M}_wide.csv'
		)

	records = [
		time_quicksorts(size)
		for size in sizes
		for _ in range(reps)
	]
	write_wide_csv(records, output_path)
	print(f'{len(records)} measurements -> {output_path}')
	return records

### Benchmark

In [2]:
data = run_benchmark(reps=5, sizes=[100, 1_000, 10_000, 100_000, 1_000_000])

25 measurements -> /Users/simon/ENSIMAG/M2R-ParallelQuicksort/data/mightbesimon.local_2026-11-10/measurements_15:06_wide.csv


In [3]:
data

[{'Size': 100, 'Seq': 1.2e-05, 'Par': 0.001894, 'Libc': 8e-06},
 {'Size': 100, 'Seq': 1e-05, 'Par': 0.002456, 'Libc': 8e-06},
 {'Size': 100, 'Seq': 9e-06, 'Par': 0.001411, 'Libc': 8e-06},
 {'Size': 100, 'Seq': 9e-06, 'Par': 0.001886, 'Libc': 7e-06},
 {'Size': 100, 'Seq': 9e-06, 'Par': 0.00175, 'Libc': 7e-06},
 {'Size': 1000, 'Seq': 0.000108, 'Par': 0.008125, 'Libc': 9e-05},
 {'Size': 1000, 'Seq': 0.00012, 'Par': 0.009377, 'Libc': 8.2e-05},
 {'Size': 1000, 'Seq': 8.6e-05, 'Par': 0.00724, 'Libc': 7.9e-05},
 {'Size': 1000, 'Seq': 8.4e-05, 'Par': 0.007832, 'Libc': 7.7e-05},
 {'Size': 1000, 'Seq': 7.8e-05, 'Par': 0.007856, 'Libc': 7.3e-05},
 {'Size': 10000, 'Seq': 0.000982, 'Par': 0.015878, 'Libc': 0.000854},
 {'Size': 10000, 'Seq': 0.00096, 'Par': 0.012272, 'Libc': 0.000858},
 {'Size': 10000, 'Seq': 0.001011, 'Par': 0.014126, 'Libc': 0.00086},
 {'Size': 10000, 'Seq': 0.000965, 'Par': 0.012255, 'Libc': 0.00086},
 {'Size': 10000, 'Seq': 0.000983, 'Par': 0.01078, 'Libc': 0.000866},
 {'Size': 